In [8]:
from pathlib import Path
from pprint import pprint

from llmgrader.mcp.server import (
    create_config_skeleton,
    explain_config,
    scan_repo_for_config_inputs,
    validate_config_xml,
)
from llmgrader.mcp.blind_user_llm import run_blind_user_llm

repo_root = Path.cwd().resolve().parent
probability_repo = repo_root / "tests" / "fixtures" / "probability_repo"

print(f"repo_root = {repo_root}")
print(f"probability_repo = {probability_repo}")
print(f"exists = {probability_repo.exists()}")

repo_root = C:\Users\sdran\Documents\repos\llmgrader
probability_repo = C:\Users\sdran\Documents\repos\llmgrader\tests\fixtures\probability_repo
exists = True


# MCP scratchpad

This notebook is a local playground for the `llmgrader` MCP config-authoring helpers.

Suggested flow:
1. Inspect the helper guidance.
2. Scan a fixture repo for likely unit XML files.
3. Build a draft `llmgrader_config.xml`.
4. Validate the generated XML.
5. Optionally run the live OpenAI blind-user harness.

In [9]:
guidance = explain_config()
scan_result = scan_repo_for_config_inputs(workspace_root=str(probability_repo))

print("Guidance summary:")
print(guidance["summary"])
print()
print("Required sections:")
pprint(guidance["required_sections"])
print()
print("Scan result:")
pprint(scan_result)

Guidance summary:
llmgrader_config.xml defines course metadata, unit XML files to include in the package, and optional asset files/directories.

Required sections:
{'course': {'description': 'Metadata for the packaged course.',
            'required_fields': ['name', 'term']},
 'units': {'description': 'List of unit XML files to include in the package.',
           'minimum_items': 1,
           'required_fields': ['name', 'source', 'destination']}}

Scan result:
{'asset_directories': ['figures'],
 'unit_xml_candidates': ['units/combinatorics.xml',
                         'units/random_variables.xml'],
 'workspace_root': 'C:\\Users\\sdran\\Documents\\repos\\llmgrader\\tests\\fixtures\\probability_repo'}


In [10]:
units = [
    {
        "name": Path(relative_path).stem,
        "source": relative_path,
        "destination": Path(relative_path).name,
    }
    for relative_path in scan_result["unit_xml_candidates"]
]
assets = [{"source": "figures", "destination": "probability_assets"}]

xml_text = create_config_skeleton(
    course_name="Probability",
    term="Fall 2026",
    units=units,
    assets=assets,
)
validation_result = validate_config_xml(
    config_xml=xml_text,
    workspace_root=str(probability_repo),
)

print(xml_text)
print()
pprint(validation_result)

<llmgrader>
  <course>
    <name>Probability</name>
    <term>Fall 2026</term>
  </course>
  <units>
    <unit>
      <name>combinatorics</name>
      <source>units/combinatorics.xml</source>
      <destination>combinatorics.xml</destination>
    </unit>
    <unit>
      <name>random_variables</name>
      <source>units/random_variables.xml</source>
      <destination>random_variables.xml</destination>
    </unit>
  </units>
  <assets>
    <asset>
      <source>figures</source>
      <destination>probability_assets</destination>
    </asset>
  </assets>
</llmgrader>

{'errors': [], 'valid': True, 'warnings': []}


In [11]:
# Optional live call. Requires OPENAI_API_KEY to be set in the environment.
# This reloads the module so notebook runs pick up recent code changes.

import importlib
import os

import llmgrader.mcp.blind_user_llm as blind_user_llm

importlib.reload(blind_user_llm)

prompt_to_run = globals().get(
    "selected_prompt",
    "Create a starter unit XML for a probability unit with one partial-credit question and validate it.",
)
print("Prompt to run:")
print(prompt_to_run)
print()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("OPENAI_API_KEY is not set for this notebook kernel.")
    print("Set it in the environment and rerun this cell.")
else:
    result = blind_user_llm.run_blind_user_llm(
        prompt=prompt_to_run,
        workspace_root=str(probability_repo),
        model="gpt-4.1",
        api_key=api_key,
        max_round_trips=6,
        verbose=True,
    )

    print("Final text:")
    print(result["final_text"])
    print()
    print("Tool calls made:")
    pprint(result["tool_calls"])

Prompt to run:
Create a starter unit XML for a probability unit with one partial-credit question about computing P(A union B) from P(A), P(B), and P(A intersection B). Include parts, a partial-credit rubric, and validate the XML before presenting it.

Prompt
Create a starter unit XML for a probability unit with one partial-credit question about computing P(A union B) from P(A), P(B), and P(A intersection B). Include parts, a partial-credit rubric, and validate the XML before presenting it.



[04/21/26 08:37:36] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 400   ]8;id=103756;file://c:\Users\sdran\Documents\repos\chat-env\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=786912;file://c:\Users\sdran\Documents\repos\chat-env\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             Bad Request"                                                                          

BadRequestError: Error code: 400 - {'error': {'message': "Invalid schema for function 'create_unit_xml_skeleton': In context=('properties', 'questions', 'type', '0', 'items', 'properties', 'rubrics', 'items'), 'required' is required to be supplied and to be an array including every key in properties. Missing 'part'.", 'type': 'invalid_request_error', 'param': 'tools[6].parameters', 'code': 'invalid_function_parameters'}}

In [12]:
import importlib
import os

import llmgrader.mcp.blind_user_llm as blind_user_llm

prompt_examples = {
    "config_probability": "Create a skeleton course config for a probability class using the XML files already in this workspace.",
    "unit_probability_binary": (
        "Create a starter unit XML for a probability unit with one binary-graded question. "
        "The question should ask students to state the sample space for two fair coin tosses. "
        "Include a simple binary rubric and validate the XML before presenting it."
    ),
    "unit_probability_partial": (
        "Create a starter unit XML for a probability unit with one partial-credit question about computing "
        "P(A union B) from P(A), P(B), and P(A intersection B). Include parts, a partial-credit rubric, "
        "and validate the XML before presenting it."
    ),
    "rubric_help": (
        "Explain the rubric rules for a partial-credit probability question and show a minimal rubric example "
        "for a problem about expected value."
    ),
}

prompt_to_run = (
    "Create a unit XML on two random variables." 
    "Add a single problem where there are random variables X and Y uniformly distributed on the region 0 <= x <= 1, 0 <= y <= x, "
     "students have to compute p(x) in part (a) and E(x) in part (b).  Include a partial-credit rubric with at least one rubric per part. Validate the XML before presenting it."
)

print("Prompt to run:")
print(prompt_to_run)
print()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("OPENAI_API_KEY is not set for this notebook kernel.")
    print("Set it in the environment and rerun this cell.")
else:
    result = blind_user_llm.run_blind_user_llm(
        prompt=prompt_to_run,
        workspace_root=str(probability_repo),
        model="gpt-4.1",
        api_key=api_key,
        max_round_trips=6,
        verbose=True,
    )

    print("Final text:")
    print(result["final_text"])
    print()
    print("Tool calls made:")
    pprint(result["tool_calls"])

Prompt to run:
Create a unit XML on two random variables.Add a single problem where there are random variables X and Y uniformly distributed on the region 0 <= x <= 1, 0 <= y <= x, students have to compute p(x) in part (a) and E(x) in part (b).  Include a partial-credit rubric with at least one rubric per part. Validate the XML before presenting it.

Prompt
Create a unit XML on two random variables.Add a single problem where there are random variables X and Y uniformly distributed on the region 0 <= x <= 1, 0 <= y <= x, students have to compute p(x) in part (a) and E(x) in part (b).  Include a partial-credit rubric with at least one rubric per part. Validate the XML before presenting it.



[04/21/26 08:37:38] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 400   ]8;id=389822;file://c:\Users\sdran\Documents\repos\chat-env\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=930829;file://c:\Users\sdran\Documents\repos\chat-env\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             Bad Request"                                                                          

BadRequestError: Error code: 400 - {'error': {'message': "Invalid schema for function 'create_unit_xml_skeleton': In context=('properties', 'questions', 'type', '0', 'items', 'properties', 'rubrics', 'items'), 'required' is required to be supplied and to be an array including every key in properties. Missing 'part'.", 'type': 'invalid_request_error', 'param': 'tools[6].parameters', 'code': 'invalid_function_parameters'}}